# Ajuste supervisado de Qwen2.5-0.5B con instrucciones en español

Cuaderno del proyecto del bloque 2 del curso **Modelos de Lenguaje: del Transformer al Agente**
(`gustavoai.dev/cursos/llm-agents/proyecto-sft-colab`).

Aquí se hace lo que el navegador no puede: el ajuste supervisado que le diste al mini-GPT con diez
pares, en la lección del mismo nombre, sobre un modelo abierto de verdad. El modelo es
**Qwen2.5-0.5B**, la versión *base*: preentrenada y nada más, sin ningún ajuste para obedecer. Los
pares son unos 3 200, escritos en español por personas.

El bucle está escrito a mano, sin la clase `Trainer` de la biblioteca, y la pérdida también: es
$\mathcal{L}_{\text{SFT}}$, la entropía cruzada sobre los tokens de la respuesta. La única línea que
hace algo que no has escrito tú es `backward()`.

**Antes de ejecutar nada:** *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)*.
La celda de la sección 2 comprueba si te ha tocado una.

## 1. Versiones

Fijadas a propósito: unos pesos, unos datos y una versión de la biblioteca son las tres partes de un
resultado reproducible, y el modelo y los datos se cargan más abajo en un *commit* fijo del Hub por la
misma razón. `torch`, `pandas` y `pyarrow` no se fijan: vienen con Colab, y reinstalar `torch` es la forma
más rápida de quedarse sin GPU. Lo que sí se hace es dejar escrito con cuáles se ejecutó.

In [ ]:
!pip install -q transformers==5.17.0 tokenizers==0.23.3 huggingface_hub==1.33.0

In [ ]:
import torch, transformers, huggingface_hub, pandas

print("torch          ", torch.__version__)
print("transformers   ", transformers.__version__)
print("huggingface_hub", huggingface_hub.__version__)
print("pandas         ", pandas.__version__)

## 2. ¿Hay GPU?

El ajuste completo pide unos 12 GB de memoria y unos $2 \times 10^{15}$ FLOPs: en la T4 gratuita son
minutos. Sin GPU, el cuaderno no termina; la cuenta está en la lección.

In [ ]:
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("memoria: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
    dispositivo = "cuda"
else:
    print("Sin GPU. Entorno de ejecución -> Cambiar tipo de entorno de ejecución -> GPU.")
    dispositivo = "cpu"

## 3. El modelo y su tokenizador

En un *commit* fijo del Hub, el de `REVISION`, no en «el último»: si alguien cambia el modelo, este
cuaderno sigue cargando el mismo. Los pesos vienen guardados en 16 bits y se cargan en `float32`, cuatro bytes por
parámetro, que es en lo que se entrenan aquí.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODELO, REVISION = "Qwen/Qwen2.5-0.5B", "060db6499f32faf8b98477b0a26969ef7d8b9987"

tok = AutoTokenizer.from_pretrained(MODELO, revision=REVISION)
modelo = AutoModelForCausalLM.from_pretrained(MODELO, revision=REVISION, dtype=torch.float32)
modelo.to(dispositivo)
print()

### La cuenta de la lección, comprobada

Lo que hay que mover y lo que ocupa moverlo. AdamW guarda dos medias móviles por parámetro, como el
`Adam` del mini-GPT, así que entrenar pide cuatro números por parámetro: el peso, su gradiente y las
dos medias. Y eso antes de guardar una sola activación.

In [ ]:
cfg = modelo.config
N = sum(p.numel() for p in modelo.parameters())
tabla = cfg.vocab_size * cfg.hidden_size

print("L=%d  d_model=%d  h=%d  d_ff=%d  |V|=%d" % (
    cfg.num_hidden_layers, cfg.hidden_size, cfg.num_attention_heads,
    cfg.intermediate_size, cfg.vocab_size))
print("parámetros          : %13s" % f"{N:,}")
print("tabla de embeddings : %13s   (%.1f %%; atada a la salida: %s)" % (
    f"{tabla:,}", 100 * tabla / N, cfg.tie_word_embeddings))
print("los pesos, float32  : %6.2f GB" % (4 * N / 1e9))
print("pesos + gradiente + dos medias móviles: %.2f GB" % (16 * N / 1e9))

## 4. La plantilla, a mano

ChatML, la plantilla de los modelos Qwen: cada mensaje abre con `<|im_start|>`, el rol escrito como
texto y un salto de línea, y cierra con `<|im_end|>`. Es la del mini-GPT con otra ortografía: allí
había un token por rol, aquí uno solo para abrir y el rol detrás, en tokens corrientes.

La función devuelve los ids y la máscara de la lección sobre la plantilla de chat: 1 en el contenido del
asistente y en su `<|im_end|>`, 0 en todo lo demás.

In [ ]:
INI, FIN = tok.convert_tokens_to_ids("<|im_start|>"), tok.convert_tokens_to_ids("<|im_end|>")
SALTO = tok.encode("\n")

def plantilla(mensajes):
    ids, mascara = [], []
    for rol, contenido in mensajes:
        cabeza = [INI] + tok.encode(rol + "\n")
        cuerpo = tok.encode(contenido) + [FIN]
        ids += cabeza + cuerpo + SALTO
        mascara += [0] * len(cabeza) + [int(rol == "assistant")] * len(cuerpo) + [0]
    return ids, mascara

def prompt(instruccion):
    # Lo que el modelo lee antes de responder: el mensaje y el turno del asistente, abierto.
    return plantilla([("user", instruccion)])[0] + [INI] + tok.encode("assistant\n")

ids, mascara = plantilla([("user", "¿De qué color es la nieve?"), ("assistant", "La nieve es blanca.")])
for i, m in zip(ids, mascara):
    print(m, repr(tok.decode([i])))

La biblioteca trae la misma plantilla, `apply_chat_template`. Con un mensaje de sistema escrito, las
dos dan el mismo texto. Sin él, la de la biblioteca añade uno en inglés que nadie ha escrito; la de
arriba escribe sólo lo que le das.

In [ ]:
conv = [("system", "Responde en español."), ("user", "¿De qué color es la nieve?"),
        ("assistant", "La nieve es blanca.")]
oficial = tok.apply_chat_template([{"role": r, "content": c} for r, c in conv], tokenize=False)
print("la misma plantilla:", tok.decode(plantilla(conv)[0]) == oficial)
print(repr(tok.apply_chat_template([{"role": "user", "content": "Hola"}], tokenize=False)))

### Dos tokens que el preentrenamiento no usó

En el modelo base, las filas de `<|im_start|>` y de `<|im_end|>` en la tabla de embeddings son el
mismo vector que las de los tokens que no se usaron nunca: todo lo que va de `<|im_start|>` al final
de la tabla. Mira cuántas son y cuánto se parecen.

Con la tabla atada a la salida, filas iguales dan logits iguales. Esos tokens empatan en cada
posición y se reparten a partes iguales la masa que reciben, así que ninguno puede pasar de
1 entre el número de filas.

In [ ]:
E = modelo.get_input_embeddings().weight.detach()
filas = E[INI:]                                      # de <|im_start|> al final de la tabla
u = torch.nn.functional.normalize(filas, dim=1)
print("filas desde <|im_start|>     :", len(filas))
print("su norma                     : de %.4f a %.4f" % (filas.norm(dim=1).min(), filas.norm(dim=1).max()))
print("coseno mínimo entre dos      : %.4f" % (u @ u.T).min())
print("norma media de las anteriores: %.4f" % E[:INI].norm(dim=1).mean())
print("cota de p(<|im_end|>)        : 1/%d = %.4f" % (len(filas), 1 / len(filas)))

## 5. Antes del ajuste

Cinco instrucciones, con la plantilla delante y generación voraz, hasta `<|im_end|>` o hasta 120
tokens. La última es la de la lección sobre de dónde salen las preferencias, una pregunta
con una premisa falsa; guárdala para después.

In [ ]:
PRUEBAS = [
    "¿Cuál es la capital de Australia?",
    "Escribe una receta de tortilla de patatas en tres pasos.",
    "Dame tres consejos para dormir mejor.",
    "Explica qué es la fotosíntesis a un niño de ocho años.",
    "Estoy seguro de que la Gran Muralla china se ve desde la Luna. ¿Verdad que sí?",
]

@torch.no_grad()
def responder(instruccion, n=120):
    modelo.eval()
    x = torch.tensor([prompt(instruccion)], device=dispositivo)
    y = modelo.generate(x, max_new_tokens=n, do_sample=False, top_p=None, top_k=None,
                        temperature=None, eos_token_id=FIN, pad_token_id=FIN)[0, x.shape[1]:]
    cierra = FIN in y.tolist()
    return tok.decode(y[:-1] if cierra else y) + ("  [<|im_end|>]" if cierra else "  [sin cerrar]")

for q in PRUEBAS:
    print("###", q)
    print(responder(q), "\n")

## 6. Los pares

OpenAssistant Conversations (`OpenAssistant/oasst1`, licencia Apache 2.0): conversaciones escritas
y puntuadas por voluntarios en 2023, en 35 idiomas. De cada conversación en español se toma el primer
mensaje, que es la instrucción, y la respuesta que los voluntarios pusieron primera en su
clasificación (`rank == 0`). Fuera los mensajes borrados y los que los propios voluntarios rechazaron al repasarlos.

Las conversaciones de más de `T_MAX` tokens se descartan, no se cortan: una respuesta cortada no
llega a su `<|im_end|>`, y el modelo aprendería a no cerrarla. La parte `train` es el conjunto de
entrenamiento; la parte `validation`, el reservado, con el que se mide.

In [ ]:
import pandas as pd
from huggingface_hub import hf_hub_download

DATOS, REVISION_DATOS = "OpenAssistant/oasst1", "fdf72ae0827c1cda404aff25b6603abec9e3399b"
FICHEROS = {"train": "data/train-00000-of-00001-b42a775f407cee45.parquet",
            "validation": "data/validation-00000-of-00001-134b8fd0c89408b6.parquet"}
T_MAX = 512

def pares(parte):
    df = pd.read_parquet(hf_hub_download(DATOS, FICHEROS[parte], repo_type="dataset",
                                         revision=REVISION_DATOS))
    df = df[(df.lang == "es") & ~df.deleted & (df.review_result == True)]
    primeros = df[df.parent_id.isna()]                           # la instrucción
    mejores = df[(df.role == "assistant") & (df["rank"] == 0)]    # la respuesta preferida
    m = primeros.merge(mejores, left_on="message_id", right_on="parent_id", suffixes=("_q", "_a"))
    todos = [(q, a, *plantilla([("user", q), ("assistant", a)])) for q, a in zip(m.text_q, m.text_a)]
    return [p for p in todos if len(p[2]) <= T_MAX], len(todos)

entrenamiento, total = pares("train")
reservados, total_r = pares("validation")
cobrados = sum(sum(p[3]) for p in entrenamiento)
tokens = sum(len(p[2]) for p in entrenamiento)
print("entrenamiento: %d de %d pares caben en %d tokens" % (len(entrenamiento), total, T_MAX))
print("               %d tokens, %d cobrados (%.0f %%)" % (tokens, cobrados, 100 * cobrados / tokens))
print("reservados   : %d de %d" % (len(reservados), total_r))
q, a, _, _ = entrenamiento[0]
print("\n###", q, "\n" + a[:400])

## 7. Qué medir, antes y después

Tres números. La pérdida del ajuste en las respuestas reservadas: $\mathcal{L}_{\text{SFT}}$ sobre
pares que el modelo no entrena, en nats por token cobrado. La probabilidad media que da a
`<|im_end|>` justo donde una respuesta reservada acaba, frente a la cota de la sección 4. Y la
pérdida en las últimas 6 000 letras de *Marianela*, el final de la novela de la que la lección del
ajuste supervisado sacó el trozo con que midió el olvido del mini-GPT. Se mide en tokens de Qwen, que
no son los del mini-GPT: este número sólo se compara con el suyo de después.

`perdida` es la de entrenar: la suma de $-\log$ sobre los tokens cobrados de una conversación. La
fila $t$ apuesta por el token $t + 1$, así que su peso es la máscara corrida una posición, como el
`peso` del mini-GPT.

In [ ]:
import math, urllib.request
import torch.nn.functional as F

novela = urllib.request.urlopen("https://www.gustavoai.dev/courses/llm-agents/corpus.txt").read()
novela = novela.decode("utf-8").split("\n", 1)[1].lstrip()      # sin la línea de cabecera
R = tok.encode(novela[-6000:])

def logits(ids):
    return modelo(input_ids=torch.tensor([ids], device=dispositivo), use_cache=False).logits[0, :-1]

def perdida(ids, mascara):
    peso = torch.tensor(mascara[1:], device=dispositivo, dtype=torch.bool)
    y = torch.tensor(ids[1:], device=dispositivo)
    return F.cross_entropy(logits(ids)[peso], y[peso], reduction="sum")

@torch.no_grad()
def medir(cuando):
    modelo.eval()
    suma = cuenta = fin = 0.0
    for _, _, ids, mascara in reservados:
        lp = logits(ids).log_softmax(-1).gather(1, torch.tensor(ids[1:], device=dispositivo)[:, None])[:, 0]
        peso = torch.tensor(mascara[1:], device=dispositivo, dtype=torch.bool)
        suma, cuenta = suma - lp[peso].sum().item(), cuenta + int(peso.sum())
        fin += lp[peso][-1].exp().item()                 # el último cobrado es el <|im_end|>
    novela_nats = F.cross_entropy(logits(R), torch.tensor(R[1:], device=dispositivo)).item()
    print("%s del ajuste" % cuando)
    print("  respuestas reservadas : %.3f nats por token" % (suma / cuenta))
    print("  p(<|im_end|>) al final: %.1e de media" % (fin / len(reservados)))
    print("  Marianela             : %.3f nats por token" % novela_nats)

medir("antes")

## 8. El ajuste

El bucle de la lección sobre entrenar el mini-GPT, con el modelo de Qwen dentro: AdamW, recorte del
gradiente a norma 1 y el mismo calendario, calentamiento lineal y coseno hasta la décima parte. Una
vuelta sobre los pares, en pasos de `LOTE` conversaciones.

Dos cosas cambian. La tasa: $\eta_{\max} = 2 \times 10^{-5}$, doscientas cincuenta veces menos que la
del mini-GPT en la lección del ajuste supervisado, porque aquí hay que frenar el olvido y no
provocarlo. Y el lote: las conversaciones pasan de una en una y el gradiente se acumula, así que no
hay relleno, y cada una divide su pérdida entre los tokens cobrados de todo el lote, como `perdida`
dividía entre la suma de `peso`.

Tarda del orden de diez minutos en una T4. Cada 20 pasos imprime la pérdida media de esos pasos,
y al final, el tiempo y la memoria que llegó a ocupar la GPU.

In [ ]:
import random, time

VUELTAS, LOTE, ETA_MAX, S_CAL = 1, 16, 2e-5, 10
S = VUELTAS * (len(entrenamiento) // LOTE)

def eta(s):
    return ETA_MAX * min(1, s / S_CAL) * (0.1 + 0.9 * (1 + math.cos(math.pi * s / S)) / 2)

opt = torch.optim.AdamW(modelo.parameters(), lr=ETA_MAX, betas=(0.9, 0.999), weight_decay=0.01)
random.seed(0)
s, t0, suma, cuenta = 0, time.time(), 0.0, 0
if dispositivo == "cuda":
    torch.cuda.reset_peak_memory_stats()
modelo.train()
for vuelta in range(VUELTAS):
    orden = random.sample(entrenamiento, len(entrenamiento))
    for i in range(0, len(orden) - LOTE + 1, LOTE):
        lote, s = orden[i:i + LOTE], s + 1
        n = sum(sum(m[1:]) for _, _, _, m in lote)              # los tokens cobrados del lote
        for g in opt.param_groups:
            g["lr"] = eta(s)
        for _, _, ids, mascara in lote:
            L = perdida(ids, mascara)
            (L / n).backward()                                    # se acumula en .grad
            suma, cuenta = suma + L.item(), cuenta + sum(mascara[1:])
        torch.nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
        opt.step()
        opt.zero_grad(set_to_none=True)
        if s % 20 == 0 or s == S:
            print("paso %3d de %d   pérdida %.3f   eta %.1e   %4.1f min" % (
                s, S, suma / cuenta, eta(s), (time.time() - t0) / 60))
            suma, cuenta = 0.0, 0
if dispositivo == "cuda":
    print("memoria máxima de la GPU: %.1f GB" % (torch.cuda.max_memory_allocated() / 1e9))

## 9. Después del ajuste

Los mismos tres números y las mismas cinco instrucciones. Compara cada uno con el de antes: si el
modelo cierra ahora sus respuestas, qué ha pasado con la cota de la sección 4, cuánto ha subido la
pérdida en la novela y qué contesta a la pregunta de la Muralla.

In [ ]:
medir("después")

In [ ]:
for q in PRUEBAS:
    print("###", q)
    print(responder(q), "\n")

### Tres instrucciones reservadas, con la respuesta del voluntario al lado

Ninguna de las dos es «la buena»: una la escribió una persona y la otra un modelo que ha leído miles
como ella.

In [ ]:
for q, a, _, _ in reservados[:3]:
    print("###", q)
    print("--- el voluntario:\n" + a[:500])
    print("--- el modelo:\n" + responder(q, n=200), "\n")

## 10. Tus propias instrucciones

Cámbialas por lo que quieras. Prueba alguna cuya respuesta puedas comprobar.

In [ ]:
for q in ["¿Cuántas patas tiene una araña?",
          "Resume en dos frases qué es un modelo de lenguaje.",
          "¿Quién escribió Marianela?"]:
    print("###", q)
    print(responder(q, n=200), "\n")

## 11. Guardar

Los pesos ajustados, con su tokenizador al lado: sin él no valen nada, porque es el que sabe qué
entrada del vocabulario corresponde a cada fila de la tabla. En `float32` ocupan unos 2 GB.

In [ ]:
modelo.save_pretrained("qwen-ajustado")
tok.save_pretrained("qwen-ajustado")
!du -sh qwen-ajustado

---

### Qué tocar a partir de aquí

Cada cambio, desde la celda del modelo de la sección 3, para partir otra vez del modelo base.

- **`ETA_MAX`**. Súbelo a `2e-4` y mira la novela: es el olvido catastrófico, medido.
- **`entrenamiento`**. Quédate con 300 pares (`entrenamiento = entrenamiento[:300]`, después de la
  sección 6) y compara las respuestas. ¿Cuánto de lo que ha cambiado es la forma?
- **`VUELTAS`**. Con 3, la pérdida en los pares de entrenamiento sigue bajando. Mira qué hace la de
  los reservados.
- **Si la memoria no da**, en otra GPU o con otro modelo, la salida habitual es LoRA (*low-rank adaptation*): congelar
  los pesos y entrenar sólo, al lado de cada matriz, dos matrices estrechas cuyo producto se le suma.
  Queda fuera del curso; la biblioteca `peft` lo hace.